# Step 3: Model Training
Train ML models to predict HOMO, LUMO, and Optical Bandgap

In [ ]:
# Install required libraries
!pip install pandas numpy scikit-learn tensorflow xgboost -q

In [ ]:
# Upload files from previous steps
from google.colab import files
print("Upload: Data_Final_merged.xlsx and features_morgan.npy")
uploaded = files.upload()

Upload: Data_Final_merged.xlsx and features_morgan.npy


Saving features_morgan.npy to features_morgan.npy
Saving Data_Final_merged.xlsx to Data_Final_merged.xlsx


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("STEP 3: MODEL TRAINING")
print("="*70)

STEP 3: MODEL TRAINING


In [ ]:
# Load data and features
print("\n[1/6] Loading data and features...")
df = pd.read_excel('Data_Final_merged.xlsx')
X = np.load('features_morgan.npy')

y_homo = df['HOMO_A'].values
y_lumo = df['LUMO_A'].values
y_eg = df['EgA_opt'].values

print(f"Features shape: {X.shape}")
print(f"Samples: {len(y_homo)}")


[1/6] Loading data and features...
Features shape: (1571, 2048)
Samples: 1571


In [ ]:
# Train/test split
print("\n[2/6] Creating train/test split (80/20)...")
X_train, X_test, y_homo_train, y_homo_test = train_test_split(
    X, y_homo, test_size=0.2, random_state=42
)
_, _, y_lumo_train, y_lumo_test = train_test_split(
    X, y_lumo, test_size=0.2, random_state=42
)
_, _, y_eg_train, y_eg_test = train_test_split(
    X, y_eg, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} molecules")
print(f"Test set: {X_test.shape[0]} molecules")


[2/6] Creating train/test split (80/20)...
Training set: 1256 molecules
Test set: 315 molecules


In [ ]:
# Dictionary to store results
results = {}

In [ ]:
# [3/6] Train Linear Regression (Baseline)
print("\n[3/6] Training Linear Regression models (baseline)...")
lr_models = {}

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg', y_eg_train, y_eg_test)]:
    lr = LinearRegression()
    lr.fit(X_train, y_tr)
    y_pred = lr.predict(X_test)

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    lr_models[prop] = lr
    results[f'LR_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"  {prop}: R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[3/6] Training Linear Regression models (baseline)...
  HOMO: R²=-0.143, MAE=0.091, RMSE=0.140
  LUMO: R²=-0.275, MAE=0.118, RMSE=0.165
  Eg: R²=-0.045, MAE=0.091, RMSE=0.138


In [ ]:
# [4/6] Train Support Vector Regression with hyperparameter tuning
print("\n[4/6] Training SVR models with hyperparameter tuning...")
print("  (This may take 5-10 minutes)")
svr_models = {}

param_grid = {
    'C': [1, 10, 100],
    'gamma': [0.001, 0.01, 0.1],
    'epsilon': [0.01, 0.1]
}

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg', y_eg_train, y_eg_test)]:
    print(f"  Tuning {prop}...")
    svr = SVR(kernel='rbf')
    grid_search = GridSearchCV(svr, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=0)
    grid_search.fit(X_train, y_tr)

    best_svr = grid_search.best_estimator_
    y_pred = best_svr.predict(X_test)

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    svr_models[prop] = best_svr
    results[f'SVR_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"    Best params: {grid_search.best_params_}")
    print(f"    R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[4/6] Training SVR models with hyperparameter tuning...
  (This may take 5-10 minutes)
  Tuning HOMO...
    Best params: {'C': 1, 'epsilon': 0.01, 'gamma': 0.001}
    R²=0.317, MAE=0.077, RMSE=0.108
  Tuning LUMO...
    Best params: {'C': 1, 'epsilon': 0.1, 'gamma': 0.01}
    R²=0.195, MAE=0.103, RMSE=0.131
  Tuning Eg...
    Best params: {'C': 1, 'epsilon': 0.01, 'gamma': 0.001}
    R²=0.357, MAE=0.070, RMSE=0.109


In [ ]:
# [5/6] Train Neural Networks (single output)
print("\n[5/6] Training Neural Network models (single output)...")
print("  (This may take 5-10 minutes)")
nn_models = {}

def create_nn_model(input_dim):
    model = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse', metrics=['mae'])
    return model

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg', y_eg_train, y_eg_test)]:
    print(f"  Training {prop}...")
    model = create_nn_model(X_train.shape[1])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    )

    history = model.fit(
        X_train, y_tr, epochs=100, batch_size=32,
        validation_split=0.2, callbacks=[early_stop], verbose=0
    )

    y_pred = model.predict(X_test, verbose=0).flatten()

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    nn_models[prop] = model
    results[f'NN_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"    R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[5/6] Training Neural Network models (single output)...
  (This may take 5-10 minutes)
  Training HOMO...
    R²=-12.091, MAE=0.412, RMSE=0.474
  Training LUMO...
    R²=-6.393, MAE=0.345, RMSE=0.398
  Training Eg...
    R²=-3.726, MAE=0.266, RMSE=0.294


In [ ]:
# [6/6] Train Multi-Output Neural Network
print("\n[6/6] Training Multi-Output Neural Network...")
y_train_multi = np.column_stack([y_homo_train, y_lumo_train, y_eg_train])
y_test_multi = np.column_stack([y_homo_test, y_lumo_test, y_eg_test])

model_multi = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(3)  # 3 outputs: HOMO, LUMO, Eg
])

model_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse', metrics=['mae']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history_multi = model_multi.fit(
    X_train, y_train_multi, epochs=100, batch_size=32,
    validation_split=0.2, callbacks=[early_stop], verbose=0
)

y_pred_multi = model_multi.predict(X_test, verbose=0)

for i, prop in enumerate(['HOMO', 'LUMO', 'Eg']):
    y_pred = y_pred_multi[:, i]
    y_te = y_test_multi[:, i]

    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    results[f'NN_Multi_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"  {prop}: R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")


[6/6] Training Multi-Output Neural Network...
  HOMO: R²=-5.410, MAE=0.244, RMSE=0.331
  LUMO: R²=-1.812, MAE=0.186, RMSE=0.245
  Eg: R²=-0.126, MAE=0.103, RMSE=0.144


In [ ]:
# [7/8] Train Random Forest
from sklearn.ensemble import RandomForestRegressor

print("\n[7/8] Training Random Forest models...")
print("  (This may take 3-5 minutes)")
rf_models = {}

rf_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 0.3]
}

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg',   y_eg_train,   y_eg_test)]:
    print(f"  Tuning RF for {prop}...")
    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    grid_search = GridSearchCV(rf, rf_param_grid, cv=3,
                               scoring='r2', n_jobs=-1, verbose=0)
    grid_search.fit(X_train, y_tr)
    best_rf = grid_search.best_estimator_

    y_pred = best_rf.predict(X_test)
    r2   = r2_score(y_te, y_pred)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    rf_models[prop] = best_rf
    results[f'RF_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"    Best params: {grid_search.best_params_}")
    print(f"    {prop} → R²={r2:.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

print("\n✓ Random Forest training complete")


[7/8] Training Random Forest models...
  (This may take 3-5 minutes)
  Tuning RF for HOMO...
    Best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 300}
    HOMO → R²=0.384  MAE=0.0750  RMSE=0.1027
  Tuning RF for LUMO...
    Best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 300}
    LUMO → R²=0.219  MAE=0.0976  RMSE=0.1293
  Tuning RF for Eg...
    Best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 100}
    Eg → R²=0.383  MAE=0.0719  RMSE=0.1063

✓ Random Forest training complete


In [ ]:
# [8/8] Train Gradient Boosting (XGBoost)
# If xgboost is not installed, we fall back to sklearn GradientBoostingRegressor
try:
    import xgboost as xgb
    USE_XGB = True
    print("\n[8/8] Training XGBoost models...")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    USE_XGB = False
    print("\n[8/8] Training Gradient Boosting models (sklearn)...")
    print("  Tip: install xgboost for faster training → !pip install xgboost -q")

print("  (This may take 5-10 minutes)")
gb_models = {}

if USE_XGB:
    gb_param_grid = {
        'n_estimators': [200, 500],
        'max_depth': [4, 6],
        'learning_rate': [0.05, 0.1],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.5, 0.8]
    }
else:
    gb_param_grid = {
        'n_estimators': [200, 500],
        'max_depth': [3, 5],
        'learning_rate': [0.05, 0.1],
        'subsample': [0.8, 1.0]
    }

for prop, y_tr, y_te in [('HOMO', y_homo_train, y_homo_test),
                          ('LUMO', y_lumo_train, y_lumo_test),
                          ('Eg',   y_eg_train,   y_eg_test)]:
    print(f"  Tuning GB for {prop}...")
    if USE_XGB:
        estimator = xgb.XGBRegressor(random_state=42, n_jobs=-1,
                                      tree_method='hist', verbosity=0)
    else:
        estimator = GradientBoostingRegressor(random_state=42)

    grid_search = GridSearchCV(estimator, gb_param_grid, cv=3,
                               scoring='r2', n_jobs=-1, verbose=0)
    grid_search.fit(X_train, y_tr)
    best_gb = grid_search.best_estimator_

    y_pred = best_gb.predict(X_test)
    r2   = r2_score(y_te, y_pred)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    label = 'XGB' if USE_XGB else 'GB'
    gb_models[prop] = best_gb
    results[f'{label}_{prop}'] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"    Best params: {grid_search.best_params_}")
    print(f"    {prop} → R²={r2:.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

print("\n✓ Gradient Boosting training complete")


[8/8] Training XGBoost models...
  (This may take 5-10 minutes)
  Tuning GB for HOMO...
    Best params: {'colsample_bytree': 0.5, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 1.0}
    HOMO → R²=0.365  MAE=0.0766  RMSE=0.1043
  Tuning GB for LUMO...
    Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 1.0}
    LUMO → R²=0.207  MAE=0.0980  RMSE=0.1304
  Tuning GB for Eg...
    Best params: {'colsample_bytree': 0.5, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}
    Eg → R²=0.385  MAE=0.0723  RMSE=0.1062

✓ Gradient Boosting training complete


In [ ]:
# Display results summary (all models)
print("\n" + "="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)

comparison_data = []
for model_name, metrics in results.items():
    parts = model_name.split('_')
    model_type = parts[0] if len(parts) == 2 else '_'.join(parts[:2])
    prop = parts[-1]
    comparison_data.append({
        'Model': model_type,
        'Property': prop,
        'R²': metrics['R2'],
        'MAE': metrics['MAE'],
        'RMSE': metrics['RMSE']
    })

comparison_df = pd.DataFrame(comparison_data)
# Sort by property then R² descending for easy reading
comparison_df = comparison_df.sort_values(['Property', 'R²'], ascending=[True, False])
print(comparison_df.to_string(index=False))



MODEL PERFORMANCE SUMMARY
   Model Property         R²      MAE     RMSE
     XGB       Eg   0.385100 0.072333 0.106176
      RF       Eg   0.383338 0.071866 0.106328
     SVR       Eg   0.357441 0.070480 0.108538
      LR       Eg  -0.044938 0.091364 0.138411
NN_Multi       Eg  -0.125833 0.102746 0.143668
      NN       Eg  -3.726158 0.266283 0.294360
      RF     HOMO   0.384259 0.074982 0.102702
     XGB     HOMO   0.365048 0.076598 0.104292
     SVR     HOMO   0.316682 0.077337 0.108192
      LR     HOMO  -0.142704 0.091227 0.139910
NN_Multi     HOMO  -5.410484 0.243643 0.331381
      NN     HOMO -12.090503 0.412254 0.473543
      RF     LUMO   0.219183 0.097608 0.129310
     XGB     LUMO   0.206555 0.098039 0.130351
     SVR     LUMO   0.195071 0.102695 0.131291
      LR     LUMO  -0.275455 0.118028 0.165268
NN_Multi     LUMO  -1.811512 0.185584 0.245373
      NN     LUMO  -6.393407 0.344888 0.397905


In [ ]:
# Identify best models
print("\n" + "="*70)
print("BEST MODELS BY PROPERTY")
print("="*70)

for prop in ['HOMO', 'LUMO', 'Eg']:
    prop_results = comparison_df[comparison_df['Property'] == prop]
    best_idx = prop_results['R²'].idxmax()
    best_model = prop_results.loc[best_idx]
    print(f"{prop}: {best_model['Model']} (R²={best_model['R²']:.3f}, MAE={best_model['MAE']:.3f})")


BEST MODELS BY PROPERTY
HOMO: RF (R²=0.384, MAE=0.075)
LUMO: RF (R²=0.219, MAE=0.098)
Eg: XGB (R²=0.385, MAE=0.072)


In [ ]:
# Save all model files
print("\n" + "="*70)
print("SAVING MODELS")
print("="*70)

with open('models_lr.pkl', 'wb') as f:
    pickle.dump(lr_models, f)
print("✓ Saved: models_lr.pkl")

with open('models_svr.pkl', 'wb') as f:
    pickle.dump(svr_models, f)
print("✓ Saved: models_svr.pkl")

with open('models_rf.pkl', 'wb') as f:
    pickle.dump(rf_models, f)
print("✓ Saved: models_rf.pkl")

with open('models_gb.pkl', 'wb') as f:
    pickle.dump(gb_models, f)
print("✓ Saved: models_gb.pkl")

for prop, model in nn_models.items():
    model.save(f'model_nn_{prop}.keras')
    print(f"✓ Saved: model_nn_{prop}.keras")

model_multi.save('model_nn_multi.keras')
print("✓ Saved: model_nn_multi.keras")

comparison_df.to_csv('model_comparison.csv', index=False)
print("✓ Saved: model_comparison.csv")



SAVING MODELS
✓ Saved: models_lr.pkl
✓ Saved: models_svr.pkl
✓ Saved: models_rf.pkl
✓ Saved: models_gb.pkl
✓ Saved: model_nn_HOMO.keras
✓ Saved: model_nn_LUMO.keras
✓ Saved: model_nn_Eg.keras
✓ Saved: model_nn_multi.keras
✓ Saved: model_comparison.csv


In [ ]:
# Download all model files
from google.colab import files
files.download('models_lr.pkl')
files.download('models_svr.pkl')
files.download('models_rf.pkl')
files.download('models_gb.pkl')
files.download('model_nn_multi.keras')
files.download('model_comparison.csv')

print("\n✓ STEP 3 COMPLETE - All 5 models trained and saved")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ STEP 3 COMPLETE - All 5 models trained and saved
